# C2 · Group leakage: the same patient in training and in validation

**Type**: Break the model · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Measure the effect of repeated measurements on the performance estimate: compare `KFold` and `GroupKFold` for a flexible model
and for a rigid one, and show that the model "recognizes" the patient from their measurements.

## Dataset

`ds.patient_measurements()`: 60 patients × 10 measurements = 600 rows, columns `patient`, `m1`, `m2`, `m3`, `outcome`. The measurements are
almost constant per patient (a fingerprint) and the outcome depends mostly on an individual characteristic that the measurements do not explain.
In production the model will be used on **never-seen** patients.

## Educational purpose

Understand that leakage depends on the combination of **data structure** (non-independent rows) and **model capacity**
(memorizing): a random forest suffers from it enormously, a linear regression hardly at all.

## Task

1. For `RandomForestRegressor(100, random_state=0)` and `LinearRegression()` compute the mean R² with `KFold(5, shuffle=True, random_state=0)` and with `GroupKFold(5)` (with `groups=g`). Build `table` (rows: models; columns: `random`, `group`, `difference`).
2. With `KFold(5, shuffle=True, random_state=0)` compute `kfold_shared_share`: the fraction of validation rows (mean over the folds) whose patient also appears in the training set. Compute the same for `GroupKFold` in `group_shared_share`.
3. Compute `patient_accuracy`: with `KNeighborsClassifier(1)` and random `KFold`, how well the **patient** (the id) is recognized from the three measurements alone. It is the proof that the measurements are a fingerprint.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, KFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier

p = ds.patient_measurements()
X, y, g = p[["m1", "m2", "m3"]], p["outcome"], p["patient"]
kf = KFold(5, shuffle=True, random_state=0)
gkf = GroupKFold(5)

## Your work

In [ ]:
models = {"random forest": RandomForestRegressor(100, random_state=0), "linear regression": LinearRegression()}
rows = {}
for name, m in models.items():
    ...
table = pd.DataFrame(rows).T
table.round(3)

In [ ]:
def shared_share(splitter, groups=None):
    ...

kfold_shared_share = ...
group_shared_share = ...
patient_accuracy = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`cross_val_score(m, X, y, cv=kf, scoring="r2").mean()` and `cross_val_score(m, X, y, cv=gkf, groups=g, scoring="r2").mean()`. For the share: `for tr, va in splitter.split(X, y, groups): np.isin(g.iloc[va], g.iloc[tr]).mean()`, then the mean. Recognition: `cross_val_score(KNeighborsClassifier(1), X, g, cv=kf)`.'):
    check(list(table.columns) == ["random", "group", "difference"] and set(table.index) == {"random forest", "linear regression"},
          "table with the two models and the three columns", "columns: random, group, difference")
    check(table.loc["random forest", "random"] > 0.6 and table.loc["random forest", "group"] < 0,
          "the random forest looks great with random K-fold and becomes worse than the mean with GroupKFold", "random > 0.6, group < 0 for the random forest")
    check(table.loc["random forest", "difference"] > 4 * table.loc["linear regression", "difference"] > 0,
          "the effect of the leakage is much larger for the flexible model (over 4 times that of the linear model)",
          "the forest must have a random-group difference much larger than that of the linear regression")
    check(kfold_shared_share > 0.99 and group_shared_share == 0, "K-fold: almost all the validation patients are also in the training set; GroupKFold: never",
          "with KFold the patient appears in the training set; with GroupKFold the groups are disjoint")
    check(patient_accuracy > 0.8, "the measurements identify the patient with over 80% accuracy (against 1.7% by chance)", "the three measurements are a fingerprint of the patient")
done()

---
The commented solution is in `solutions/C_break_the_model/C2_group_leakage_patients.ipynb`. Open it after you pass the check, or after you have really tried.